# Constructing Automation & Augmentation Indices
## Bloom's Taxonomy Approach

Using Anthropic Economic Index task penetration data + O*NET task-to-occupation crosswalk.

**Key insight:** O*NET task verbs signal cognitive complexity. Lower-order verbs
(calculate, process, sort) → tasks AI can fully automate. Higher-order verbs
(analyze, evaluate, design) → tasks where AI augments human judgment.

**Pipeline:**
1. Load & merge data
2. spaCy: extract root verb from each task
3. Classify verbs into Bloom's Taxonomy levels
4. Score tasks: automation (Bloom 1-3) vs augmentation (Bloom 4-6)
5. Validate
6. Aggregate to occupation level → export CSV for Stata

In [ ]:
import pandas as pd
import numpy as np
import spacy
from collections import Counter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('All imports OK')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
penetration = pd.read_csv('../raw/Anthropic Task/task_penetration.csv')
crosswalk = pd.read_csv('../raw/Crosswalks/onet_task_statements.csv')

print(f'Penetration: {penetration.shape[0]} tasks')
print(f'Crosswalk:   {crosswalk.shape[0]} task-occupation pairs')
print(f'Unique occupations in crosswalk: {crosswalk["O*NET-SOC Code"].nunique()}')
print()
print('Penetration columns:', penetration.columns.tolist())
print('Crosswalk columns:', crosswalk.columns.tolist())

In [ ]:
# ── Merge penetration scores onto crosswalk ──────────────────────────────────
# Crosswalk col is 'Task', penetration col is 'task'
merged = crosswalk.merge(penetration, left_on='Task', right_on='task', how='left')

# Check merge quality
n_matched = merged['penetration'].notna().sum()
n_total = len(merged)
print(f'Matched: {n_matched} / {n_total} ({100*n_matched/n_total:.1f}%)')
print(f'Unmatched: {n_total - n_matched}')

# Fill unmatched with 0 penetration (task not observed in Claude usage)
merged['penetration'] = merged['penetration'].fillna(0.0)

# Keep relevant columns
df = merged[['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type',
             'Incumbents Responding', 'penetration']].copy()

print(f'\nWorking dataset: {len(df)} rows')
print(f'Non-zero penetration: {(df["penetration"] > 0).sum()}')
df.head()

## Step 1: Extract root verb from each task statement

In [ ]:
# ── spaCy verb extraction ────────────────────────────────────────────────────
nlp = spacy.load('en_core_web_sm')

def extract_root_verb(text):
    """Extract the root verb from an O*NET task statement."""
    doc = nlp(text)
    # Strategy 1: find the ROOT token if it's a verb
    for token in doc:
        if token.dep_ == 'ROOT' and token.pos_ == 'VERB':
            return token.lemma_.lower()
    # Strategy 2: first verb in the sentence
    for token in doc:
        if token.pos_ == 'VERB':
            return token.lemma_.lower()
    # Fallback: first token lowered
    return doc[0].lemma_.lower() if len(doc) > 0 else ''

# Get unique tasks (avoid re-parsing duplicates)
unique_tasks = df['Task'].unique()
print(f'Unique task statements: {len(unique_tasks)}')

# Extract verbs
task_to_verb = {}
for i, task in enumerate(unique_tasks):
    task_to_verb[task] = extract_root_verb(task)
    if (i + 1) % 5000 == 0:
        print(f'  Parsed {i+1}/{len(unique_tasks)}')

df['root_verb'] = df['Task'].map(task_to_verb)

# Top verbs
verb_counts = Counter(task_to_verb.values())
print(f'\nUnique verbs: {len(verb_counts)}')
print('Top 25 verbs:')
for v, c in verb_counts.most_common(25):
    print(f'  {v:20s} {c:5d}')

## Step 2: Classify verbs via Bloom's Taxonomy

Levels 1-3 (Remember / Understand / Apply) → **Automation**: tasks AI can execute independently.

Levels 4-6 (Analyze / Evaluate / Create) → **Augmentation**: tasks where AI assists human judgment.

In [ ]:
# ── Bloom's Taxonomy verb classification ──────────────────────────────────────
# Levels 1-3: Automation (Remember, Understand, Apply)
# Levels 4-6: Augmentation (Analyze, Evaluate, Create)

bloom = {
    # Level 1: Remember
    'list': 1, 'define': 1, 'identify': 1, 'recall': 1, 'name': 1,
    'state': 1, 'recognize': 1, 'label': 1, 'locate': 1, 'memorize': 1,
    'retrieve': 1, 'repeat': 1, 'duplicate': 1,

    # Level 2: Understand
    'describe': 2, 'explain': 2, 'summarize': 2, 'classify': 2,
    'interpret': 2, 'paraphrase': 2, 'translate': 2, 'discuss': 2,
    'report': 2, 'inform': 2, 'communicate': 2, 'present': 2,
    'illustrate': 2, 'demonstrate': 2, 'clarify': 2, 'distinguish': 2,

    # Level 3: Apply
    'calculate': 3, 'compute': 3, 'execute': 3, 'implement': 3,
    'process': 3, 'operate': 3, 'prepare': 3, 'administer': 3,
    'compile': 3, 'sort': 3, 'file': 3, 'schedule': 3, 'apply': 3,
    'use': 3, 'perform': 3, 'complete': 3, 'produce': 3, 'carry': 3,
    'deliver': 3, 'provide': 3, 'serve': 3, 'conduct': 3,
    'maintain': 3, 'install': 3, 'repair': 3, 'assemble': 3,
    'clean': 3, 'load': 3, 'transport': 3, 'distribute': 3,
    'collect': 3, 'gather': 3, 'record': 3, 'enter': 3,
    'input': 3, 'verify': 3, 'check': 3, 'inspect': 3,
    'measure': 3, 'count': 3, 'stock': 3, 'store': 3,
    'pack': 3, 'ship': 3, 'route': 3, 'assign': 3,
    'coordinate': 3, 'arrange': 3, 'organize': 3, 'set': 3,
    'adjust': 3, 'calibrate': 3, 'configure': 3, 'update': 3,
    'issue': 3, 'submit': 3, 'forward': 3, 'post': 3,
    'transfer': 3, 'copy': 3, 'scan': 3, 'print': 3,
    'type': 3, 'draft': 3, 'fill': 3, 'obtain': 3,
    'purchase': 3, 'order': 3, 'request': 3, 'supply': 3,
    'follow': 3, 'comply': 3, 'enforce': 3, 'ensure': 3,
    'protect': 3, 'secure': 3, 'monitor': 3, 'track': 3,
    'log': 3, 'document': 3, 'notify': 3, 'alert': 3,
    'respond': 3, 'answer': 3, 'greet': 3, 'assist': 3,
    'help': 3, 'support': 3, 'accompany': 3, 'escort': 3,
    'attend': 3, 'observe': 3, 'watch': 3, 'patrol': 3,
    'drive': 3, 'pilot': 3, 'steer': 3, 'navigate': 3,
    'mix': 3, 'blend': 3, 'cook': 3, 'bake': 3,
    'weld': 3, 'cut': 3, 'drill': 3, 'grind': 3,
    'paint': 3, 'sand': 3, 'sew': 3, 'trim': 3,
    'lift': 3, 'move': 3, 'place': 3, 'position': 3,
    'remove': 3, 'replace': 3, 'attach': 3, 'connect': 3,
    'fasten': 3, 'secure': 3, 'tighten': 3, 'lubricate': 3,
    'feed': 3, 'water': 3, 'groom': 3, 'bathe': 3,
    'dress': 3, 'change': 3, 'dispose': 3, 'discard': 3,
    'take': 3, 'accept': 3, 'receive': 3, 'pick': 3,
    'select': 3, 'choose': 3, 'mark': 3, 'tag': 3,
    'weigh': 3, 'test': 3, 'sample': 3, 'run': 3,
    'start': 3, 'stop': 3, 'turn': 3, 'open': 3,
    'close': 3, 'lock': 3, 'unlock': 3, 'activate': 3,
    'deactivate': 3, 'reset': 3, 'reboot': 3, 'troubleshoot': 3,
    'read': 3, 'review': 3, 'proofread': 3, 'edit': 3,
    'transcribe': 3, 'dictate': 3, 'index': 3, 'catalog': 3,
    'register': 3, 'enroll': 3, 'book': 3, 'reserve': 3,
    'confirm': 3, 'approve': 3, 'authorize': 3, 'sign': 3,
    'stamp': 3, 'seal': 3, 'certify': 3, 'license': 3,
    'direct': 3, 'instruct': 3, 'train': 3, 'teach': 3,
    'tutor': 3, 'coach': 3, 'mentor': 3, 'guide': 3,
    'lead': 3, 'manage': 3, 'supervise': 3, 'oversee': 3,
    'delegate': 3, 'assign': 3, 'dispatch': 3, 'deploy': 3,
    'hire': 3, 'recruit': 3, 'interview': 3, 'screen': 3,
    'orient': 3, 'onboard': 3, 'terminate': 3, 'dismiss': 3,
    'discipline': 3, 'counsel': 3, 'mediate': 3, 'arbitrate': 3,
    'negotiate': 3, 'advocate': 3, 'represent': 3, 'lobby': 3,
    'promote': 3, 'market': 3, 'advertise': 3, 'sell': 3,
    'solicit': 3, 'canvass': 3, 'contact': 3, 'call': 3,
    'email': 3, 'write': 3, 'compose': 3, 'correspond': 3,
    'meet': 3, 'confer': 3, 'consult': 3, 'collaborate': 3,
    'participate': 3, 'contribute': 3, 'volunteer': 3,
    'fundraise': 3, 'donate': 3, 'sponsor': 3,

    # Level 4: Analyze
    'analyze': 4, 'compare': 4, 'examine': 4, 'investigate': 4,
    'differentiate': 4, 'diagnose': 4, 'audit': 4, 'assess': 4,
    'research': 4, 'study': 4, 'survey': 4, 'probe': 4,
    'explore': 4, 'map': 4, 'model': 4, 'simulate': 4,
    'forecast': 4, 'predict': 4, 'estimate': 4, 'project': 4,
    'quantify': 4, 'benchmark': 4, 'profile': 4, 'characterize': 4,
    'correlate': 4, 'decompose': 4, 'deconstruct': 4, 'parse': 4,
    'categorize': 4, 'segment': 4, 'stratify': 4, 'prioritize': 4,
    'rank': 4, 'rate': 4, 'score': 4, 'grade': 4,
    'detect': 4, 'screen': 4, 'scan': 4, 'identify': 4,
    'determine': 4, 'ascertain': 4, 'establish': 4, 'discover': 4,

    # Level 5: Evaluate
    'evaluate': 5, 'judge': 5, 'recommend': 5, 'advise': 5,
    'appraise': 5, 'critique': 5, 'justify': 5,
    'decide': 5, 'conclude': 5, 'resolve': 5, 'rule': 5,
    'adjudicate': 5, 'referee': 5, 'umpire': 5,
    'approve': 5, 'reject': 5, 'accept': 5, 'deny': 5,
    'endorse': 5, 'sanction': 5, 'validate': 5, 'invalidate': 5,
    'weigh': 5, 'balance': 5, 'consider': 5, 'deliberate': 5,
    'debate': 5, 'argue': 5, 'defend': 5, 'challenge': 5,

    # Level 6: Create
    'design': 6, 'develop': 6, 'plan': 6, 'formulate': 6,
    'propose': 6, 'create': 6, 'invent': 6, 'innovate': 6,
    'conceive': 6, 'envision': 6, 'imagine': 6, 'ideate': 6,
    'construct': 6, 'build': 6, 'engineer': 6, 'architect': 6,
    'devise': 6, 'craft': 6, 'author': 6, 'originate': 6,
    'pioneer': 6, 'initiate': 6, 'launch': 6, 'found': 6,
    'establish': 6, 'institute': 6, 'generate': 6, 'synthesize': 6,
    'integrate': 6, 'combine': 6, 'merge': 6, 'unify': 6,
    'adapt': 6, 'modify': 6, 'revise': 6, 'refine': 6,
    'improve': 6, 'enhance': 6, 'optimize': 6, 'upgrade': 6,
    'transform': 6, 'reshape': 6, 'reimagine': 6, 'reinvent': 6,
    'customize': 6, 'tailor': 6, 'personalize': 6, 'specialize': 6,
    'program': 6, 'code': 6, 'script': 6, 'configure': 6,
    'choreograph': 6, 'orchestrate': 6, 'curate': 6, 'compile': 6,
}

# Classify each task's root verb
def get_bloom(verb):
    return bloom.get(verb, 0)  # 0 = unclassified

task_bloom = {t: get_bloom(v) for t, v in task_to_verb.items()}

# Coverage check
classified = sum(1 for b in task_bloom.values() if b > 0)
total = len(task_bloom)
print(f'Classified: {classified} / {total} ({100*classified/total:.1f}%)')
print()

# Unclassified verbs (top 30)
unclassified_verbs = Counter(
    v for t, v in task_to_verb.items() if task_bloom[t] == 0
)
print(f'Unclassified unique verbs: {len(unclassified_verbs)}')
print('Top 30 unclassified:')
for v, c in unclassified_verbs.most_common(30):
    print(f'  {v:20s} {c:5d}')

# Distribution across Bloom levels
bloom_dist = Counter(task_bloom.values())
for level in sorted(bloom_dist.keys()):
    label = {0: 'Unclassified', 1: 'Remember', 2: 'Understand',
             3: 'Apply', 4: 'Analyze', 5: 'Evaluate', 6: 'Create'}[level]
    print(f'  Level {level} ({label:14s}): {bloom_dist[level]:5d} tasks')

## Step 3: Score tasks as automation vs augmentation

In [ ]:
# ── Build task-level scores ───────────────────────────────────────────────────
pen_lookup = penetration.set_index('task')['penetration'].to_dict()

task_scores = pd.DataFrame({
    'task': list(task_to_verb.keys()),
    'root_verb': list(task_to_verb.values()),
    'bloom_level': [task_bloom[t] for t in task_to_verb.keys()],
})

# Auto = Bloom 1-3, Aug = Bloom 4-6
task_scores['is_auto'] = task_scores['bloom_level'].between(1, 3).astype(int)
task_scores['is_aug']  = task_scores['bloom_level'].between(4, 6).astype(int)

# Merge penetration
task_scores['penetration'] = task_scores['task'].map(pen_lookup).fillna(0.0)

print(f'Task scores: {len(task_scores)}')
print(f'  Automation tasks (Bloom 1-3): {task_scores["is_auto"].sum()}')
print(f'  Augmentation tasks (Bloom 4-6): {task_scores["is_aug"].sum()}')
print(f'  Unclassified (Bloom 0): {(task_scores["bloom_level"] == 0).sum()}')
print()
print(f'Mean penetration — auto tasks:  {task_scores.loc[task_scores["is_auto"]==1, "penetration"].mean():.4f}')
print(f'Mean penetration — aug tasks:   {task_scores.loc[task_scores["is_aug"]==1, "penetration"].mean():.4f}')
print(f'Mean penetration — unclassified: {task_scores.loc[task_scores["bloom_level"]==0, "penetration"].mean():.4f}')
task_scores.head(10)

## Step 4: Validate Bloom classification

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) Penetration distribution by category
for cat, color, label in [(1, '#d73027', 'Automation (Bloom 1-3)'),
                            (2, '#2166ac', 'Augmentation (Bloom 4-6)'),
                            (0, '#888888', 'Unclassified')]:
    if cat == 0:
        mask = task_scores['bloom_level'] == 0
    elif cat == 1:
        mask = task_scores['is_auto'] == 1
    else:
        mask = task_scores['is_aug'] == 1
    subset = task_scores.loc[mask & (task_scores['penetration'] > 0), 'penetration']
    if len(subset) > 0:
        axes[0].hist(subset, bins=40, alpha=0.5, color=color, label=label)
axes[0].set_xlabel('Penetration (non-zero only)')
axes[0].set_ylabel('Count')
axes[0].set_title('(a) AI penetration by Bloom category')
axes[0].legend(fontsize=8)

# (b) Bloom level distribution
levels = sorted(task_scores['bloom_level'].unique())
level_names = {0: 'Unclass.', 1: 'Remember', 2: 'Understand',
               3: 'Apply', 4: 'Analyze', 5: 'Evaluate', 6: 'Create'}
level_counts = [task_scores[task_scores['bloom_level'] == l].shape[0] for l in levels]
colors = ['#888888'] + ['#d73027']*3 + ['#2166ac']*3
axes[1].barh([level_names[l] for l in levels], level_counts, color=colors[:len(levels)])
axes[1].set_xlabel('Number of tasks')
axes[1].set_title('(b) Tasks by Bloom level')
axes[1].invert_yaxis()

# (c) Mean penetration by Bloom level
mean_pens = [task_scores.loc[task_scores['bloom_level'] == l, 'penetration'].mean()
             for l in levels]
axes[2].barh([level_names[l] for l in levels], mean_pens, color=colors[:len(levels)])
axes[2].set_xlabel('Mean penetration')
axes[2].set_title('(c) Mean AI penetration by Bloom level')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('../output/graphs/descriptives/bloom_validation.pdf',
            bbox_inches='tight')
plt.show()

# Top tasks per category
for label, mask in [('AUTOMATION (Bloom 1-3)', task_scores['is_auto'] == 1),
                     ('AUGMENTATION (Bloom 4-6)', task_scores['is_aug'] == 1)]:
    top = task_scores.loc[mask].nlargest(10, 'penetration')
    print(f'\n--- {label}: top 10 by penetration ---')
    for _, row in top.iterrows():
        print(f'  [{row["root_verb"]:12s}] pen={row["penetration"]:.3f}  {row["task"][:80]}')

## Step 5: Aggregate to occupation level

In [ ]:
# ── Merge task scores with crosswalk → occupation level ────────────────────
occ_tasks = crosswalk[['O*NET-SOC Code', 'Title', 'Task', 'Task Type',
                        'Incumbents Responding']].copy()

occ_tasks = occ_tasks.merge(task_scores[['task', 'is_auto', 'is_aug',
                                          'bloom_level', 'penetration']],
                             left_on='Task', right_on='task', how='left')

# Fill unmatched
occ_tasks['is_auto'] = occ_tasks['is_auto'].fillna(0)
occ_tasks['is_aug'] = occ_tasks['is_aug'].fillna(0)
occ_tasks['penetration'] = occ_tasks['penetration'].fillna(0.0)

# Weight: Task Type (Core=2, Supplemental=1) × Incumbents Responding
occ_tasks['weight'] = np.where(occ_tasks['Task Type'] == 'Core', 2.0, 1.0)
occ_tasks['weight'] *= occ_tasks['Incumbents Responding'].fillna(1.0)

# Aggregation:
# AutoIndex  = weighted sum of (penetration × is_auto) / total weight
# AugIndex   = weighted sum of (penetration × is_aug)  / total weight
def agg_occ(g):
    w = g['weight']
    pen = g['penetration']
    total_w = w.sum()
    return pd.Series({
        'automation_index':    (pen * g['is_auto'] * w).sum() / total_w,
        'augmentation_index':  (pen * g['is_aug']  * w).sum() / total_w,
        'mean_penetration':    (pen * w).sum() / total_w,
        'n_tasks': len(g),
        'n_tasks_nonzero': (pen > 0).sum(),
        'pct_auto_tasks': g['is_auto'].mean(),
        'pct_aug_tasks': g['is_aug'].mean(),
    })

occ_index = occ_tasks.groupby(['O*NET-SOC Code', 'Title']).apply(agg_occ).reset_index()

print(f'Occupation-level index: {len(occ_index)} occupations')
print(occ_index[['automation_index', 'augmentation_index', 'mean_penetration']].describe())
print()
print('Top 10 by automation_index:')
print(occ_index.nlargest(10, 'automation_index')[['Title', 'automation_index',
      'augmentation_index', 'mean_penetration']].to_string(index=False))
print()
print('Top 10 by augmentation_index:')
print(occ_index.nlargest(10, 'augmentation_index')[['Title', 'augmentation_index',
      'automation_index', 'mean_penetration']].to_string(index=False))

In [ ]:
# ── Export for Stata ──────────────────────────────────────────────────────────
occ_index['soc_code'] = occ_index['O*NET-SOC Code'].str.replace('.00', '', regex=False)

export_cols = ['soc_code', 'O*NET-SOC Code', 'Title', 'automation_index',
               'augmentation_index', 'mean_penetration', 'n_tasks',
               'pct_auto_tasks', 'pct_aug_tasks']

occ_index[export_cols].to_csv('../output/data/anthropic_occ_index.csv', index=False)
print(f'Exported {len(occ_index)} occupations to output/data/anthropic_occ_index.csv')

# Task-level scores
task_scores.to_csv('../output/data/anthropic_task_scores.csv', index=False)
print(f'Exported {len(task_scores)} task scores to output/data/anthropic_task_scores.csv')

In [ ]:
# ── Final plots: automation vs augmentation ───────────────────────────────────

# Helper: pick one representative per corner
med_auto = occ_index['automation_index'].median()
med_aug  = occ_index['augmentation_index'].median()

corners = {
    'aug high, auto low': occ_index[
        (occ_index['augmentation_index'] > med_aug) &
        (occ_index['automation_index'] <= med_auto)
    ].nlargest(1, 'augmentation_index'),
    'auto high, aug low': occ_index[
        (occ_index['automation_index'] > med_auto) &
        (occ_index['augmentation_index'] <= med_aug)
    ].nlargest(1, 'automation_index'),
    'both high': occ_index[
        (occ_index['automation_index'] > med_auto) &
        (occ_index['augmentation_index'] > med_aug)
    ].nlargest(1, 'mean_penetration'),
    'both low': occ_index[
        (occ_index['automation_index'] <= med_auto) &
        (occ_index['augmentation_index'] <= med_aug)
    ].nsmallest(1, 'mean_penetration'),
}

def add_corner_labels(ax):
    for _, rows in corners.items():
        for _, row in rows.iterrows():
            label = row['Title']
            if len(label) > 35:
                label = label[:33] + '…'
            ax.annotate(label,
                        xy=(row['automation_index'], row['augmentation_index']),
                        xytext=(8, 5), textcoords='offset points',
                        fontsize=7, fontstyle='italic',
                        arrowprops=dict(arrowstyle='->', color='black', lw=0.7))


# ── Version 1: plain scatter ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(occ_index['automation_index'], occ_index['augmentation_index'],
           s=20, alpha=0.5, color='#2166ac', edgecolors='none')

ax.plot([0, 0.5], [0.5, 0], color='gray', ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Automation Index')
ax.set_ylabel('Augmentation Index')
ax.set_title('Occupation-Level Automation vs Augmentation')
ax.set_xlim(-0.01, 0.45)
ax.set_ylim(-0.01, 0.38)
add_corner_labels(ax)

plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_scatter.pdf',
            bbox_inches='tight')
plt.show()


# ── Version 2: bubble size = sample share, label large bubbles ────────────────
import os
cps = pd.read_stata(os.path.join('..', 'output', 'data', 'CPS-ASEC_analysis_Dec01.dta'),
                     columns=['occsoc'])
cps['occsoc'] = cps['occsoc'].str.strip()

occ_counts = cps['occsoc'].value_counts().reset_index()
occ_counts.columns = ['soc_code', 'n_workers']
occ_counts['share'] = occ_counts['n_workers'] / occ_counts['n_workers'].sum()

plot_df = occ_index.merge(occ_counts, on='soc_code', how='left')
plot_df['share'] = plot_df['share'].fillna(0)
plot_df['bubble_size'] = plot_df['share'] / plot_df['share'].max() * 500

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(plot_df['automation_index'], plot_df['augmentation_index'],
           s=plot_df['bubble_size'], alpha=0.4, color='#2166ac',
           edgecolors='#2166ac', linewidths=0.3)

ax.plot([0, 0.5], [0.5, 0], color='gray', ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Automation Index')
ax.set_ylabel('Augmentation Index')
ax.set_title('Automation vs Augmentation (bubble = sample share)')
ax.set_xlim(-0.01, 0.45)
ax.set_ylim(-0.01, 0.38)

# Corner labels
add_corner_labels(ax)

# Label large bubbles in the middle region (auto > 0.12, aug > 0.02, top by share)
mid_region = plot_df[
    (plot_df['automation_index'] > 0.12) &
    (plot_df['augmentation_index'] > 0.02)
].nlargest(6, 'share')

for _, row in mid_region.iterrows():
    label = row['Title']
    if len(label) > 35:
        label = label[:33] + '…'
    ax.annotate(label,
                xy=(row['automation_index'], row['augmentation_index']),
                xytext=(8, -8), textcoords='offset points',
                fontsize=6.5, fontstyle='italic', color='#333333',
                arrowprops=dict(arrowstyle='->', color='#555555', lw=0.5))

plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_bubble.pdf',
            bbox_inches='tight')
plt.show()
print('Done. Saved both versions.')

## Step 6: Major occupation group (soc_gr) aggregation

In [ ]:
# ── Aggregate to soc_gr level, weighted by CPS population ─────────────────────
import os

# Load full CPS with occupation codes
cps_full = pd.read_stata(
    os.path.join('..', 'output', 'data', 'CPS-ASEC_analysis_Dec01.dta'),
    columns=['occsoc', 'soc_gr', 'soc_gr_str']
)
cps_full['occsoc'] = cps_full['occsoc'].str.strip()

# Count workers per detailed occupation
occ_pop = cps_full.groupby('occsoc').agg(
    n_workers=('occsoc', 'size'),
    soc_gr=('soc_gr', 'first'),
    soc_gr_str=('soc_gr_str', 'first'),
).reset_index()
occ_pop.rename(columns={'occsoc': 'soc_code'}, inplace=True)

# Merge indices onto population counts
gr_df = occ_pop.merge(occ_index[['soc_code', 'automation_index', 'augmentation_index',
                                   'mean_penetration']],
                       on='soc_code', how='inner')

print(f'Matched {len(gr_df)} detailed occupations to CPS')

# Worker-weighted aggregation to soc_gr
def wavg(g, col):
    return np.average(g[col], weights=g['n_workers'])

gr_agg = gr_df.groupby(['soc_gr', 'soc_gr_str']).apply(
    lambda g: pd.Series({
        'automation_index': wavg(g, 'automation_index'),
        'augmentation_index': wavg(g, 'augmentation_index'),
        'mean_penetration': wavg(g, 'mean_penetration'),
        'n_workers': g['n_workers'].sum(),
    })
).reset_index()

gr_agg['share'] = gr_agg['n_workers'] / gr_agg['n_workers'].sum()
gr_agg['bubble_size'] = gr_agg['share'] / gr_agg['share'].max() * 800

# Shorten group names
gr_agg['label'] = gr_agg['soc_gr_str'].str.replace(' Occupations', '', regex=False)
gr_agg['label'] = gr_agg['label'].str.replace(' occupations', '', regex=False)

print(f'\n{len(gr_agg)} major occupation groups')
print(gr_agg[['label', 'automation_index', 'augmentation_index',
              'share']].sort_values('automation_index', ascending=False).to_string(index=False))


# ── Version 3: soc_gr scatter (plain, labeled) ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(gr_agg['automation_index'], gr_agg['augmentation_index'],
           s=60, alpha=0.7, color='#2166ac', edgecolors='white', linewidths=0.5)

for _, row in gr_agg.iterrows():
    label = row['label']
    if len(label) > 28:
        label = label[:26] + '…'
    ax.annotate(label,
                xy=(row['automation_index'], row['augmentation_index']),
                xytext=(5, 4), textcoords='offset points',
                fontsize=6, alpha=0.85)

ax.set_xlabel('Automation Index (worker-weighted)')
ax.set_ylabel('Augmentation Index (worker-weighted)')
ax.set_title('Major Occupation Groups: Automation vs Augmentation')

plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_socgr.pdf',
            bbox_inches='tight')
plt.show()


# ── Version 4: soc_gr bubble (size = population share, labeled) ──────────────
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(gr_agg['automation_index'], gr_agg['augmentation_index'],
           s=gr_agg['bubble_size'], alpha=0.4, color='#2166ac',
           edgecolors='#2166ac', linewidths=0.5)

for _, row in gr_agg.iterrows():
    label = row['label']
    if len(label) > 28:
        label = label[:26] + '…'
    ax.annotate(label,
                xy=(row['automation_index'], row['augmentation_index']),
                xytext=(5, 4), textcoords='offset points',
                fontsize=6, alpha=0.85)

ax.set_xlabel('Automation Index (worker-weighted)')
ax.set_ylabel('Augmentation Index (worker-weighted)')
ax.set_title('Major Occupation Groups (bubble = population share)')

plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_socgr_bubble.pdf',
            bbox_inches='tight')
plt.show()

# Export soc_gr level index
gr_agg[['soc_gr', 'soc_gr_str', 'automation_index', 'augmentation_index',
        'mean_penetration', 'n_workers', 'share']].to_csv(
    '../output/data/anthropic_socgr_index.csv', index=False)
print('Exported soc_gr index to output/data/anthropic_socgr_index.csv')